# 📋 Pipeline de Transcription et Résumé Syndical

Ce notebook traite un fichier audio de 10h (réunion syndicat/direction) avec :
- Transcription avec WhisperX (GPU) — modèle `large-v3`
- Diarisation des locuteurs via Pyannote 3.1
- Génération d'un Procès-Verbal administratif avec `qwen3.5:9b` via Ollama

## ⚠️ Prérequis
Installer les dépendances **une seule fois** dans ton terminal avant de lancer ce notebook :
```bash
pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
pip install git+https://github.com/m-bain/whisperx.git ollama pydub av
```

## ⚠️ Gestion de la VRAM (16 Go — RTX 2000 Ada)
WhisperX et Ollama ne tournent **pas** simultanément.
Un nettoyage strict de la VRAM sépare les deux étapes.

---
## ⚙️ CELLULE 1 : Variables globales

In [ ]:
import torch, whisperx, gc, ollama, os
import av, numpy as np

# --- PATCH : remplace ffmpeg.exe par PyAV (contourne AppLocker/WDAC) ---
def load_audio_direct(file_path: str, sr: int = 16000):
    container = av.open(file_path)
    stream = container.streams.audio[0]
    resampler = av.AudioResampler(format='s16', layout='mono', rate=sr)
    frames = []
    for frame in container.decode(stream):
        for r_frame in resampler.resample(frame):
            frames.append(r_frame.to_ndarray())
    for r_frame in resampler.resample(None):
        frames.append(r_frame.to_ndarray())
    container.close()
    if not frames:
        return np.zeros((0,), dtype=np.float32)
    return np.concatenate(frames, axis=1).flatten().astype(np.float32) / 32768.0

# Injection du patch APRÈS import de whisperx
import whisperx.audio
whisperx.audio.load_audio = load_audio_direct
whisperx.load_audio = load_audio_direct
print("✅ Patch PyAV actif — ffmpeg.exe contourné")

# --- CONFIGURATION ---
AUDIO_DIR          = "sources"          # Dossier contenant tous les fichiers audio
SUPPORTED_FORMATS  = (".m4a", ".mp3", ".wav", ".flac", ".ogg", ".mp4")
HF_TOKEN           = os.getenv('HF_TOKEN')
DEVICE             = "cuda"
MODEL_WHISPER      = "large-v3"
LANGUAGE           = "fr"               # Langue audio (évite auto-détection)
COMPUTE_TYPE       = "int8_float16"
WHISPER_BATCH_SIZE = 8
MODEL_OLLAMA       = "qwen3.5:9b"
OUTPUT_DIR         = "resultats"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

# Détection des fichiers audio
audio_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(SUPPORTED_FORMATS)
])

if not audio_files:
    raise FileNotFoundError(f"❌ Aucun fichier audio trouvé dans '{AUDIO_DIR}'. Formats supportés : {SUPPORTED_FORMATS}")

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU : {torch.cuda.get_device_name(0)} ({vram:.1f} Go VRAM)")
else:
    raise SystemError("❌ CUDA non détecté. Vérifiez vos pilotes NVIDIA.")

print(f"✅ Configuration prête — Modèle Whisper : {MODEL_WHISPER} ({LANGUAGE}) | Ollama : {MODEL_OLLAMA}")
print(f"📁 {len(audio_files)} fichier(s) audio détecté(s) :")
for f in audio_files:
    print(f"   - {f}")

---
## 🎙️ CELLULE 2 : Transcription et Diarisation (étape GPU intensive)

In [ ]:
from whisperx.diarize import DiarizationPipeline

print(f"🚀 Démarrage pipeline sur {DEVICE}...")
print(f"📁 {len(audio_files)} fichier(s) à traiter...")

# Chargement unique des modèles (évite de recharger à chaque fichier)
print(f"📥 Chargement Whisper {MODEL_WHISPER}...")
model = whisperx.load_model(MODEL_WHISPER, DEVICE, compute_type=COMPUTE_TYPE)

print("📥 Chargement modèle de diarisation (une seule fois)...")
diarize_model = DiarizationPipeline(
    model_name="pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
    device=DEVICE
)

all_segments = []

for file_idx, audio_file in enumerate(audio_files, 1):
    print(f"\n🎤 [{file_idx}/{len(audio_files)}] Traitement de {os.path.basename(audio_file)}...")

    # --- 1. TRANSCRIPTION ---
    audio = whisperx.load_audio(audio_file)
    result = model.transcribe(audio, batch_size=WHISPER_BATCH_SIZE, language=LANGUAGE)
    print(f"   ✅ Transcription terminée — langue : {result['language']}")

    # --- 2. ALIGNEMENT ---
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=DEVICE)
    result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE, return_char_alignments=False)
    del model_a
    gc.collect()
    torch.cuda.empty_cache()
    print("   ✅ Alignement terminé.")

    # --- 3. DIARISATION (modèle déjà chargé) ---
    print("   👥 Diarisation en cours...")
    diarize_segments = diarize_model(audio)
    result = whisperx.assign_word_speakers(diarize_segments, result)
    print("   ✅ Diarisation terminée.")

    file_label = os.path.splitext(os.path.basename(audio_file))[0]
    all_segments.append(f"\n\n=== FICHIER : {file_label} ===\n")
    for seg in result["segments"]:
        speaker   = seg.get("speaker", "INCONNU")
        timestamp = f"[{seg['start']:.1f}s - {seg['end']:.1f}s]"
        all_segments.append(f"{timestamp} {speaker}: {seg['text']}\n")

# Libération des modèles après tous les fichiers
del model, diarize_model
gc.collect()
torch.cuda.empty_cache()
print(f"\n🧹 VRAM libérée : {torch.cuda.memory_reserved()/1e9:.1f} Go réservés")

# --- SAUVEGARDE ---
output_path = f"{OUTPUT_DIR}/transcription_complete.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.writelines(all_segments)

total_lignes = len(all_segments)
print(f"\n💾 Transcription sauvegardée : {output_path} ({total_lignes} lignes)")
print("\n--- APERÇU DES 10 PREMIERS SEGMENTS ---")
for line in all_segments[1:11]:
    print(line.strip())

**Structure du dossier :**
```
projet/
├── sources/
│   ├── reunion_matin.m4a
│   ├── reunion_aprem.m4a
│   └── ...
├── resultats/
└── pipeline_transcription.ipynb
```

---
## 🧹 CELLULE 3 : Nettoyage STRICT de la VRAM

In [7]:
print("🧹 Nettoyage VRAM pour libérer la place à Ollama...")

for var_name in ['model', 'model_a', 'diarize_model']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print(f"✅ VRAM après nettoyage : {torch.cuda.memory_reserved()/1e9:.2f} Go réservés")
print("✅ Prêt pour Ollama.")

🧹 Nettoyage VRAM pour libérer la place à Ollama...
✅ VRAM après nettoyage : 0.05 Go réservés
✅ Prêt pour Ollama.


---
## 📝 CELLULE 4 : Génération du PV avec Ollama (qwen3.5:9b)

In [ ]:
import ollama, os, time

INPUT_FILE            = f"{OUTPUT_DIR}/transcription_complete.txt"
LINES_PER_CHUNK       = 100
OPTIONS_SEGMENT       = {"temperature": 0.1, "num_predict": 6000, "num_ctx": 8192}
OPTIONS_RESUME        = {"temperature": 0.2, "num_predict": 6000, "num_ctx": 16384}

print(f"🚀 Génération des PV avec {MODEL_OLLAMA}...")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"❌ Fichier introuvable : {INPUT_FILE}. Relance la Cellule 2 d'abord.")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    content = f.read()

# --- Découpage par fichier source ---
sections = {}
current_label = None
current_lines = []

for line in content.splitlines(keepends=True):
    if line.startswith("=== FICHIER :"):
        if current_label and current_lines:
            sections[current_label] = current_lines
        current_label = line.strip().replace("=== FICHIER :", "").replace("===", "").strip()
        current_lines = []
    else:
        if current_label:
            current_lines.append(line)

if current_label and current_lines:
    sections[current_label] = current_lines

print(f"📁 {len(sections)} fichier(s) source détecté(s) : {list(sections.keys())}")

# --- Accumulateurs pour le fichier combiné ---
all_resumes = []

def get_content(resp):
    """Récupère le contenu de la réponse Ollama (compatible dict et objet)."""
    try:
        return resp["message"]["content"]
    except TypeError:
        return resp.message.content

# --- Traitement de chaque fichier séparément ---
for file_label, lines in sections.items():
    print(f"\n{'='*60}")
    print(f"📂 Traitement de : {file_label} ({len(lines)} lignes)")
    print(f"{'='*60}")
    t_start = time.time()

    chunks = [
        "".join(lines[i:i+LINES_PER_CHUNK])
        for i in range(0, len(lines), LINES_PER_CHUNK)
    ]

    print(f"📦 {len(chunks)} segments à traiter...")

    # ===================================================================
    # ÉTAPE 1 : PV DÉTAILLÉ — un segment à la fois (livrable principal)
    # ===================================================================
    notes = []
    for i, chunk in enumerate(chunks, 1):
        print(f"📝 Rédaction segment {i}/{len(chunks)}...", end=" ", flush=True)
        t_seg = time.time()
        try:
            resp = ollama.chat(
                model=MODEL_OLLAMA,
                think=False,  # ← Désactive le mode "thinking" de Qwen3
                messages=[{"role": "user", "content": f"""Tu es un greffier chargé de retranscrire mot pour mot la substance d'une réunion syndicale.

CONSIGNES STRICTES :
1. Discours rapporté au présent (ex: "Le président rappelle que", "UFAP répond que").
2. Identifie les organisations (UFAP, FO, CGT, SPS) et fonctions (Le président, La secrétaire générale).
3. Note TOUS les arguments, même mineurs. Ne supprime rien.
4. Note précisément les chiffres, dates, noms de lieux, références réglementaires.
5. Retranscris CHAQUE prise de parole, même courte. Ne fusionne pas les interventions.
6. N'écris pas "En résumé", "Globalement" ou toute formule de synthèse.
7. Longueur attendue : au moins 300 mots par segment.

Segment ({i}/{len(chunks)}) :
{chunk}

Retranscription exhaustive :"""}],
                options=OPTIONS_SEGMENT
            )
            note = get_content(resp)
            notes.append(note)
            words = len(note.split())
            print(f"✅ ({time.time()-t_seg:.0f}s, {words} mots)")
        except Exception as e:
            print(f"⚠️ Erreur : {e}")

    # --- Sauvegarde du PV détaillé (concaténation des segments) ---
    pv_detail_path = f"{OUTPUT_DIR}/PV_DETAIL_{file_label}.txt"
    with open(pv_detail_path, "w", encoding="utf-8") as f:
        f.write(f"PROCÈS-VERBAL DÉTAILLÉ — {file_label}\n")
        f.write(f"{'='*60}\n")
        f.write(f"Généré par {MODEL_OLLAMA} | {len(chunks)} segments traités\n")
        f.write(f"{'='*60}\n\n")
        for i, note in enumerate(notes, 1):
            f.write(f"\n--- SEGMENT {i}/{len(chunks)} ---\n\n")
            f.write(note)
            f.write("\n")

    print(f"\n💾 PV DÉTAILLÉ sauvegardé : {pv_detail_path}")

    # ===================================================================
    # ÉTAPE 2 : RÉSUMÉ SYNTHÉTIQUE (points clés par groupes de segments)
    # ===================================================================
    print(f"📋 Génération du résumé synthétique...")

    GROUP_SIZE = 5
    points_cles = []
    for g in range(0, len(notes), GROUP_SIZE):
        group = notes[g:g+GROUP_SIZE]
        group_text = "\n\n".join(group)
        try:
            resp = ollama.chat(
                model=MODEL_OLLAMA,
                think=False,  # ← Désactive le mode "thinking"
                messages=[{"role": "user", "content": f"""Extrais les POINTS CLÉS de ces notes de réunion syndicale.

CONSIGNES :
- Liste chaque décision, engagement, désaccord et question restée ouverte.
- Format : liste à puces, une puce par point.
- Conserve les noms d'organisations et les chiffres exacts.
- Maximum 15 points clés pour ce groupe.

Notes (segments {g+1} à {min(g+GROUP_SIZE, len(notes))}) :
{group_text}

Points clés :"""}],
                options=OPTIONS_RESUME
            )
            points_cles.append(get_content(resp))
        except Exception as e:
            print(f"⚠️ Erreur résumé groupe {g//GROUP_SIZE+1} : {e}")

    # Sauvegarde résumé individuel
    resume_path = f"{OUTPUT_DIR}/RESUME_{file_label}.txt"
    resume_text = f"RÉSUMÉ — {file_label}\n"
    resume_text += f"{'='*60}\n"
    resume_text += f"⚠️  Ce résumé est un condensé. Le PV détaillé fait foi.\n"
    resume_text += f"    Voir : PV_DETAIL_{file_label}.txt\n"
    resume_text += f"{'='*60}\n\n"
    for pc in points_cles:
        resume_text += pc + "\n\n"

    with open(resume_path, "w", encoding="utf-8") as f:
        f.write(resume_text)

    # Accumulation pour le fichier combiné
    all_resumes.append(resume_text)

    elapsed = time.time() - t_start
    print(f"💾 Résumé sauvegardé : {resume_path}")
    print(f"⏱️  Temps total pour {file_label} : {elapsed/60:.1f} min")

# ===================================================================
# FICHIER COMBINÉ : tous les résumés dans un seul document
# ===================================================================
combined_path = f"{OUTPUT_DIR}/RESUME_COMPLET.txt"
with open(combined_path, "w", encoding="utf-8") as f:
    f.write(f"RÉSUMÉS DE TOUTES LES RÉUNIONS\n")
    f.write(f"{'='*60}\n")
    f.write(f"Généré par {MODEL_OLLAMA}\n")
    f.write(f"⚠️  Ces résumés sont des condensés. Les PV détaillés font foi.\n")
    f.write(f"{'='*60}\n\n")
    for resume in all_resumes:
        f.write(resume)
        f.write(f"\n{'─'*60}\n\n")

print(f"\n💾 Résumé combiné sauvegardé : {combined_path}")
print("\n✅ Tous les fichiers ont été traités !")
print("📁 Fichiers créés :")
for file_label in sections.keys():
    print(f"   📄 {OUTPUT_DIR}/PV_DETAIL_{file_label}.txt  (PV exhaustif)")
    print(f"   📋 {OUTPUT_DIR}/RESUME_{file_label}.txt    (résumé)")
print(f"   📋 {OUTPUT_DIR}/RESUME_COMPLET.txt         (tous les résumés)")

---
## 🏁 CELLULE 5 : Nettoyage final

In [ ]:
print("🧹 Nettoyage final...")

try:
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ VRAM finale : {torch.cuda.memory_reserved()/1e9:.2f} Go réservés")
except Exception as e:
    print(f"✅ Nettoyage terminé ({e})")

print("\n✅ Pipeline terminé avec succès !")
print("📁 Fichiers générés :")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    tag = ""
    if f.startswith("PV_DETAIL_"):
        tag = " 📄 (PV exhaustif — fait foi)"
    elif f.startswith("RESUME_"):
        tag = " 📋 (résumé rapide)"
    elif f == "transcription_complete.txt":
        tag = " 🎤 (transcription brute)"
    print(f"   - {filepath} ({size_kb:.1f} Ko){tag}")